## Manufacturing Dataset Generator

Generates a synthetic **manufacturing** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `manufacturing` | `equipment` | ~3K | `production_orders` | 100K-500K | Maintenance tracking, energy consumption, scrap cost, yield variance (70-105%) |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `manufacturing` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.manufacturing') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.manufacturing');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(99)
random.seed(99)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "manufacturing"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = datetime(2026, 3, 21)

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Equipment table (~3,000 rows) ---
equip_types = ["CNC Machine", "Injection Molder", "Conveyor Belt", "Robotic Arm", "Laser Cutter",
               "Press Brake", "Welding Station", "3D Printer", "Assembly Line", "Packaging Unit"]
manufacturers = ["Siemens", "Fanuc", "ABB", "Bosch", "Mitsubishi", "Haas", "DMG Mori", "Trumpf", "KUKA", "Mazak"]
locations = [
    "Plant A - Line 1", "Plant A - Line 2", "Plant A - Line 3", "Plant A - Line 4",
    "Plant B - Line 1", "Plant B - Line 2", "Plant B - Line 3",
    "Plant C - Line 1", "Plant C - Line 2",
    "Plant D - Line 1", "Plant D - Line 2"]

annual_hours_by_type = {
    "CNC Machine": 4500, "Injection Molder": 5500, "Conveyor Belt": 6500,
    "Robotic Arm": 5000, "Laser Cutter": 3500, "Press Brake": 3000,
    "Welding Station": 3500, "3D Printer": 4000, "Assembly Line": 6000, "Packaging Unit": 5500}

# Energy consumption per operating hour by equipment type (kWh)
energy_per_hour = {
    "CNC Machine": 15.0, "Injection Molder": 45.0, "Conveyor Belt": 5.0,
    "Robotic Arm": 8.0, "Laser Cutter": 25.0, "Press Brake": 18.0,
    "Welding Station": 12.0, "3D Printer": 3.5, "Assembly Line": 10.0, "Packaging Unit": 6.0}

NUM_EQUIPMENT = 3000
equipment = []
for i in range(1, NUM_EQUIPMENT + 1):
    equip_type = random.choice(equip_types)
    install_date = date(2012, 1, 1) + timedelta(days=random.randint(0, 5000))
    age_years = (NOW.date() - install_date).days / 365.25

    if age_years > 8:
        status = random.choices(["Operational", "Under Maintenance", "Idle", "Decommissioned"],
                                weights=[45, 20, 15, 20])[0]
    elif age_years > 5:
        status = random.choices(["Operational", "Under Maintenance", "Idle", "Decommissioned"],
                                weights=[60, 20, 13, 7])[0]
    else:
        status = random.choices(["Operational", "Under Maintenance", "Idle", "Decommissioned"],
                                weights=[80, 12, 7, 1])[0]

    base_annual = annual_hours_by_type[equip_type]
    utilization = clamp(random.gauss(0.85, 0.12), 0.4, 1.0)
    if status == "Decommissioned": utilization *= 0.5
    elif status == "Idle": utilization *= 0.6
    op_hours = int(clamp(age_years * base_annual * utilization, 500, 80000))

    base_eff = clamp(random.betavariate(5.0, 1.5) * (99.5 - 65.0) + 65.0, 65.0, 99.5)
    if status == "Decommissioned": base_eff = clamp(base_eff * random.uniform(0.70, 0.85), 65.0, 85.0)
    elif status == "Under Maintenance": base_eff = clamp(base_eff * random.uniform(0.85, 0.95), 65.0, 95.0)
    elif status == "Idle": base_eff = clamp(base_eff * random.uniform(0.80, 0.92), 65.0, 92.0)
    eff = round(base_eff, 1)

    max_maint_days = (NOW.date() - install_date).days
    if max_maint_days < 30:
        maint_offset = max_maint_days
    else:
        maint_offset = int(max_maint_days * clamp(random.betavariate(5.0, 2.0), 0.0, 1.0))
        maint_offset = max(30, maint_offset)
    last_maint = install_date + timedelta(days=maint_offset)
    if last_maint > NOW.date():
        last_maint = NOW.date() - timedelta(days=random.randint(1, 30))

    # Maintenance count: correlated with age and operating hours
    maint_count = max(1, int(clamp(age_years * random.gauss(3.5, 1.0), 1, 50)))
    # Next scheduled maintenance: 30-180 days from last
    next_maint = last_maint + timedelta(days=random.randint(30, 180))
    # Total energy consumption: operating hours * energy per hour with noise
    total_energy = round(op_hours * energy_per_hour[equip_type] * clamp(random.gauss(1.0, 0.1), 0.8, 1.2), 1)

    equipment.append(Row(
        equipment_id=i,
        equipment_name=f"{equip_type}-{random.randint(100, 999)}",
        equipment_type=equip_type,
        manufacturer=random.choice(manufacturers),
        location=random.choice(locations),
        status=status,
        install_date=install_date,
        last_maintenance_date=last_maint,
        next_maintenance_date=next_maint,
        maintenance_count=maint_count,
        operating_hours=op_hours,
        efficiency_pct=eff,
        total_energy_kwh=total_energy
    ))

equip_lookup = {e.equipment_id: e for e in equipment}

equipment_df = spark.createDataFrame(equipment)
equipment_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.equipment")
print(f"✔ Created {CATALOG_SCHEMA}.equipment ({equipment_df.count()} rows)")

# --- Production Orders table (randomized ~100K-500K rows) ---
product_names = ["Widget A", "Widget B", "Gear Assembly", "Circuit Board X1", "Bracket L",
                 "Housing Unit", "Shaft Coupling", "Valve Body", "Sensor Module", "Panel Frame",
                 "Rotor Disc", "Bearing Sleeve", "Heat Exchanger", "Filter Cartridge", "Motor Mount"]

equip_product_affinity = {
    "CNC Machine":       ["Gear Assembly", "Shaft Coupling", "Valve Body", "Bearing Sleeve", "Rotor Disc", "Bracket L"],
    "Injection Molder":  ["Housing Unit", "Widget A", "Widget B", "Panel Frame", "Filter Cartridge"],
    "Conveyor Belt":     product_names,
    "Robotic Arm":       ["Circuit Board X1", "Sensor Module", "Widget A", "Widget B", "Motor Mount"],
    "Laser Cutter":      ["Panel Frame", "Bracket L", "Rotor Disc", "Heat Exchanger"],
    "Press Brake":       ["Bracket L", "Panel Frame", "Heat Exchanger", "Motor Mount"],
    "Welding Station":   ["Heat Exchanger", "Motor Mount", "Shaft Coupling", "Valve Body"],
    "3D Printer":        ["Sensor Module", "Widget A", "Widget B", "Filter Cartridge", "Housing Unit"],
    "Assembly Line":     product_names,
    "Packaging Unit":    product_names,
}

order_statuses = ["Completed", "In Progress", "Scheduled", "On Hold", "Cancelled"]
priorities = ["Low", "Medium", "High", "Critical"]
defect_types_list = ["Dimensional", "Surface Finish", "Material", "Assembly", "Cosmetic", "Electrical", "Contamination"]

product_defect_weights = {
    "Gear Assembly":     [45, 20, 15, 12, 3, 0, 5],
    "Shaft Coupling":    [42, 22, 16, 12, 3, 0, 5],
    "Valve Body":        [40, 18, 20, 14, 3, 0, 5],
    "Bearing Sleeve":    [48, 22, 14, 8, 3, 0, 5],
    "Rotor Disc":        [44, 20, 16, 10, 5, 0, 5],
    "Widget A":          [12, 35, 10, 5, 30, 0, 8],
    "Widget B":          [12, 33, 12, 5, 30, 0, 8],
    "Housing Unit":      [15, 32, 13, 8, 25, 0, 7],
    "Filter Cartridge":  [10, 25, 28, 8, 17, 0, 12],
    "Bracket L":         [35, 15, 22, 15, 8, 0, 5],
    "Panel Frame":       [30, 20, 18, 12, 15, 0, 5],
    "Motor Mount":       [22, 10, 25, 23, 8, 5, 7],
    "Circuit Board X1":  [10, 5, 15, 30, 5, 30, 5],
    "Sensor Module":     [8, 8, 15, 28, 5, 30, 6],
    "Heat Exchanger":    [18, 12, 35, 15, 5, 5, 10],
}

# Unit cost per product for scrap cost calculation
product_unit_cost = {
    "Widget A": 2.50, "Widget B": 3.10, "Gear Assembly": 18.00, "Circuit Board X1": 45.00,
    "Bracket L": 5.00, "Housing Unit": 8.50, "Shaft Coupling": 22.00, "Valve Body": 30.00,
    "Sensor Module": 55.00, "Panel Frame": 12.00, "Rotor Disc": 25.00, "Bearing Sleeve": 15.00,
    "Heat Exchanger": 85.00, "Filter Cartridge": 6.50, "Motor Mount": 28.00,
}

# Pre-generate operators with faker names, assigned to shifts
day_operators = [fake.name() for _ in range(60)]
night_operators = [fake.name() for _ in range(35)]
swing_operators = [fake.name() for _ in range(25)]

orders = []
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
for i in range(1, NUM_EVENT_RECORDS + 1):
    start_dt = date(2024, 1, 1) + timedelta(days=random.randint(0, 730))

    eid = random.randint(1, NUM_EQUIPMENT)
    equip_info = equip_lookup[eid]
    products_for_equip = equip_product_affinity.get(equip_info.equipment_type, product_names)
    product = random.choice(products_for_equip)

    shift = random.choices(["Day", "Night", "Swing"], weights=[55, 25, 20])[0]
    if shift == "Day": operator = random.choice(day_operators)
    elif shift == "Night": operator = random.choice(night_operators)
    else: operator = random.choice(swing_operators)

    if product in ["Circuit Board X1", "Sensor Module", "Heat Exchanger"]:
        priority = random.choices(priorities, weights=[10, 30, 40, 20])[0]
    else:
        priority = random.choices(priorities, weights=[20, 40, 30, 10])[0]

    planned_qty = int(clamp(random.lognormvariate(5.5, 1.0), 50, 10000))
    cycle_time = round(clamp(random.lognormvariate(2.5, 0.8), 0.5, 240.0), 2)

    production_minutes = planned_qty * cycle_time
    production_days = max(1, int(math.ceil(production_minutes / 450)))
    production_days = min(production_days, 90)
    end_dt = start_dt + timedelta(days=production_days)

    if end_dt < NOW.date() - timedelta(days=7):
        status = random.choices(order_statuses, weights=[80, 2, 0, 5, 13])[0]
    elif start_dt > NOW.date():
        status = random.choices(order_statuses, weights=[0, 0, 75, 15, 10])[0]
    elif start_dt <= NOW.date() <= end_dt:
        status = random.choices(order_statuses, weights=[5, 65, 5, 20, 5])[0]
    else:
        status = random.choices(order_statuses, weights=[70, 10, 0, 10, 10])[0]

    # Wider actual_quantity variance: 70-105% of planned (was 80-102%)
    if status == "Completed":
        actual_qty = max(0, int(planned_qty * clamp(random.gauss(0.95, 0.08), 0.70, 1.05)))
    elif status == "In Progress":
        progress = clamp(random.betavariate(2.0, 2.0), 0.1, 0.8)
        actual_qty = max(0, int(planned_qty * progress))
    elif status == "Cancelled":
        actual_qty = max(0, int(planned_qty * random.uniform(0.0, 0.3)))
    else:
        actual_qty = 0

    defect_rate = clamp(random.betavariate(2.0, 80.0), 0.001, 0.15)
    defect_qty = int(actual_qty * defect_rate)

    if defect_qty > 0:
        weights = product_defect_weights.get(product, [15, 15, 15, 15, 15, 15, 10])
        defect_type = random.choices(defect_types_list, weights=weights)[0]
    else:
        defect_type = None

    # Scrap cost: defect quantity * unit cost
    scrap_cost = round(defect_qty * product_unit_cost.get(product, 10.0), 2)
    # Energy consumed: based on cycle time, quantity, and equipment energy rate
    energy = round(actual_qty * cycle_time / 60.0 * energy_per_hour.get(equip_info.equipment_type, 10.0) * clamp(random.gauss(1.0, 0.1), 0.8, 1.2), 2) if actual_qty > 0 else 0.0

    orders.append(Row(
        order_id=2000 + i,
        equipment_id=eid,
        product_name=product,
        order_status=status,
        priority=priority,
        shift=shift,
        start_date=start_dt,
        end_date=end_dt,
        planned_quantity=planned_qty,
        actual_quantity=actual_qty,
        defect_quantity=defect_qty,
        cycle_time_minutes=cycle_time,
        defect_type=defect_type,
        operator_name=operator,
        scrap_cost=scrap_cost,
        energy_consumption_kwh=energy
    ))

orders_df = spark.createDataFrame(orders)
orders_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.production_orders")
print(f"✔ Created {CATALOG_SCHEMA}.production_orders ({orders_df.count()} rows)")

print("\n--- Equipment (sample) ---")
display(equipment_df.limit(5))
print("\n--- Production Orders (sample) ---")
display(orders_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.manufacturing.equipment", {
    "equipment_id":           "Unique identifier for the equipment",
    "equipment_name":         "Descriptive name with model number suffix",
    "equipment_type":         "Category of equipment (e.g., CNC Machine, Robotic Arm, Laser Cutter)",
    "manufacturer":           "Equipment manufacturer name",
    "location":               "Plant and production line location (11 locations across 4 plants)",
    "status":                 "Operational status: Operational, Under Maintenance, Idle, or Decommissioned",
    "install_date":           "Date the equipment was installed (DateType)",
    "last_maintenance_date":  "Date of most recent maintenance (DateType)",
    "next_maintenance_date":  "Scheduled next maintenance date (DateType). 30-180 days after last maintenance",
    "maintenance_count":      "Total number of maintenance events since installation. Correlated with equipment age",
    "operating_hours":        "Cumulative operating hours since installation",
    "efficiency_pct":         "Current operating efficiency as a percentage (65-99.5%, beta-distributed)",
    "total_energy_kwh":       "Cumulative energy consumption in kWh. Based on operating hours and equipment-type-specific energy rates",
})

apply_comments(f"{CATALOG}.manufacturing.production_orders", {
    "order_id":               "Unique identifier for the production order",
    "equipment_id":           "Foreign key referencing equipment.equipment_id",
    "product_name":           "Name of the product being manufactured",
    "order_status":           "Order status: Completed, In Progress, Scheduled, On Hold, or Cancelled",
    "priority":               "Order priority: Low, Medium, High, or Critical",
    "shift":                  "Production shift: Day, Night, or Swing",
    "start_date":             "Planned start date (DateType)",
    "end_date":               "Planned end date (DateType)",
    "planned_quantity":       "Target production quantity (log-normal distributed)",
    "actual_quantity":        "Actual units produced. Wider variance: 70-105% of planned for completed orders",
    "defect_quantity":        "Number of defective units detected (beta-rate-based)",
    "cycle_time_minutes":     "Average cycle time per unit in minutes (log-normal distributed)",
    "defect_type":            "Type of defect: Dimensional, Surface Finish, Material, Assembly, Cosmetic, Electrical, Contamination, or NULL",
    "operator_name":          "Operator name assigned to this order (generated via Faker, shift-correlated)",
    "scrap_cost":             "Economic cost of defects: defect_quantity x product unit cost",
    "energy_consumption_kwh": "Energy consumed for this order based on actual_quantity, cycle_time, and equipment energy rate",
})

print(f"\n\u2705 All column comments applied for manufacturing schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.manufacturing') IS
'Manufacturing sample dataset with realistic statistical distributions and Faker-generated PII. Entity table: `equipment` (~3K rows). Event table: `production_orders` (100K-500K rows). Key features: Maintenance tracking, energy consumption, scrap cost, yield variance (70-105%).';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`manufacturing` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to manufacturing schema ({remove_after_value})")